# Limpeza de Dados: The Movies Dataset
Esta etapa realiza a **higienização completa** do `movies_metadata.csv`, um dataset de alta heterogeneidade com colunas em formato JSON aninhado, campos financeiros corrompidos e registros sem atividade de público. O produto desta etapa é um dataset tabular, tipado e livre de ruídos — pronto para receber a camada de inteligência do notebook de Engenharia de Features.

- **Escopo:** Descarte de colunas de baixa utilidade analítica, tratamento de nulos por limiar percentual, filtragem de registros sem engajamento e correção de tipos
- **Produto desta etapa:** `movies_cleaned.csv`

In [35]:
# Configuração do Jupyter (Autoreload)
%load_ext autoreload
%autoreload 2

# Configuração de Caminho (Path Setup)
import sys
import os

# Adiciona a pasta raiz do projeto (..) ao sistema para liberar os imports locais
sys.path.append(os.path.abspath(os.path.join('..')))


# Importação de Bibliotecas e Módulos
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

# Nossos módulos customizados da pasta src/
from src import load_data_csv

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


--- 
## Carregando Dados

In [36]:
caminho = '../data/raw/movies_dataset/movies_metadata.csv'
df_movies = load_data_csv(caminho)

Dados carregados com sucesso! Formato: (45466, 24)


---
## Aplicando Limpeza de Missing Data 

### Descarte por Alta Cardinalidade e Ausência Estrutural

Colunas como `homepage` e `tagline` combinam dois problemas que as tornam inviáveis para análise: alta cardinalidade de texto livre e taxa de preenchimento abaixo de 50%. Além de não contribuírem para nenhuma dimensão analítica do projeto, sua presença introduziria ruído nas etapas de modelagem e agregação.

- **Critério:** Colunas com ≥ 50% de nulos **e** sem valor analítico para os objetivos do projeto foram descartadas
- **Princípio:** A redução de dimensionalidade nesta etapa protege a integridade de todas as análises posteriores

In [37]:
# Dropping colunas irrelevantes para a análise que possuem alta cardinalidade de texto
# Essas colunas não são úteis para a análise e podem introduzir ruído
# Além disso, possuem 50% ou mais de valores ausentes, o que pode prejudicar a qualidade dos dados

colunas_para_dropar = ['homepage', 'tagline', 'overview','poster_path', 'status', 'imdb_id']

df_movies_limpo = df_movies.drop(columns=colunas_para_dropar)

In [38]:
df_movies_limpo.info()

<class 'pandas.DataFrame'>
RangeIndex: 45466 entries, 0 to 45465
Data columns (total 18 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   adult                  45466 non-null  str    
 1   belongs_to_collection  4494 non-null   str    
 2   budget                 45466 non-null  str    
 3   genres                 45466 non-null  str    
 4   id                     45466 non-null  str    
 5   original_language      45455 non-null  str    
 6   original_title         45466 non-null  str    
 7   popularity             45461 non-null  str    
 8   production_companies   45463 non-null  str    
 9   production_countries   45463 non-null  str    
 10  release_date           45379 non-null  str    
 11  revenue                45460 non-null  float64
 12  runtime                45203 non-null  float64
 13  spoken_languages       45460 non-null  str    
 14  title                  45460 non-null  str    
 15  video        

### Filtro de Engajamento de Público: Exclusão de Filmes sem Audiência Verificável

Filmes com zero votos não possuem sinal de recepção do público — são registros incompletos que distorceriam qualquer métrica de popularidade, média de avaliação ou análise de correlação entre orçamento e aprovação. A presença desses registros invalidaria conclusões sobre o comportamento real do mercado cinematográfico.

In [39]:
# Dropando filmes com 0 votos
df_movies_limpo = df_movies_limpo[df_movies_limpo['vote_count'] > 0]

In [40]:
numero_filmes_zero_votos = (df_movies_limpo['vote_count'] == 0).sum()
print(f'Número de filmes com 0 votos: {numero_filmes_zero_votos}')

Número de filmes com 0 votos: 0


### Integridade da Dimensão Temporal do Filme

A duração (`runtime`) é uma variável de segmentação relevante para distinguir longas-metragens de curtas e para análises comparativas de gênero. Registros com duração ausente ou igual a zero representam cadastros incompletos que não podem ser classificados em nenhuma categoria de produto cinematográfico.

In [41]:
# Dropando coluna runtime
df_movies_limpo = df_movies_limpo.dropna(subset=['runtime'])
df_movies_limpo = df_movies_limpo[df_movies_limpo['runtime'] > 0]

In [42]:
df_movies_limpo.info()

<class 'pandas.DataFrame'>
Index: 41190 entries, 0 to 45463
Data columns (total 18 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   adult                  41190 non-null  str    
 1   belongs_to_collection  4344 non-null   str    
 2   budget                 41190 non-null  str    
 3   genres                 41190 non-null  str    
 4   id                     41190 non-null  str    
 5   original_language      41184 non-null  str    
 6   original_title         41190 non-null  str    
 7   popularity             41190 non-null  str    
 8   production_companies   41190 non-null  str    
 9   production_countries   41190 non-null  str    
 10  release_date           41170 non-null  str    
 11  revenue                41190 non-null  float64
 12  runtime                41190 non-null  float64
 13  spoken_languages       41190 non-null  str    
 14  title                  41190 non-null  str    
 15  video             

### Estratégia de Tratamento de Nulos por Limiar Percentual

Para as colunas restantes, a decisão de descartar linhas versus manter registros com nulos não é arbitrária: ela é orientada pelo **impacto percentual sobre o volume total do dataset**. Colunas com taxa de ausência ≤ 0,1% têm seus registros nulos removidos sem impacto estatístico relevante, preservando a qualidade sem sacrificar volume analítico significativo.

- **Critério operacional:** Perda de dados aceitável definida em ≤ 0,1% do dataset total
- **Resultado esperado:** Máxima completude de registros nas colunas analíticas críticas

In [43]:
# Calculando os nulos de todas as colunas
nulos_por_coluna = df_movies.isnull().sum()

# Filtramndopara manter APENAS as colunas que têm buracos (> 0)
colunas_com_nulos = nulos_por_coluna[nulos_por_coluna > 0]

# Calculamos a porcentagem
percentual_nulos = (colunas_com_nulos / len(df_movies)) * 100

# Juntando tudo num DataFrame de resumo
df_resumo_nulos = pd.DataFrame({
    'Valores Nulos (Qtd)': colunas_com_nulos,
    'Perda de Dados (%)': percentual_nulos
}).sort_values(by='Perda de Dados (%)', ascending=False)

# Selecionando apenas as colunas com perda de dados menor que 0.1% para considerar o drop
df_resumo_nulos_para_dropar = df_resumo_nulos[df_resumo_nulos['Perda de Dados (%)'] < 0.1]

In [44]:
# Dropando as linhas com menos de 0.1% de perda de dados

# Criando uma lista de colunas que tem perda de dados menor ou igual a 0.1%
indices_limpos = [i.split(' (')[0] for i in df_resumo_nulos_para_dropar.index]

# Criando uma lista segura de colunas para drop, garantindo que elas existam no DataFrame
lista_segura = [coluna for coluna in indices_limpos if coluna in df_movies_limpo.columns]

# Dropando as linhas com menos de 0.1% de perda de dados
df_movies_limpo = df_movies_limpo.dropna(subset=lista_segura)

In [45]:
df_movies_limpo.info()

<class 'pandas.DataFrame'>
Index: 41184 entries, 0 to 45463
Data columns (total 18 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   adult                  41184 non-null  str    
 1   belongs_to_collection  4344 non-null   str    
 2   budget                 41184 non-null  str    
 3   genres                 41184 non-null  str    
 4   id                     41184 non-null  str    
 5   original_language      41184 non-null  str    
 6   original_title         41184 non-null  str    
 7   popularity             41184 non-null  str    
 8   production_companies   41184 non-null  str    
 9   production_countries   41184 non-null  str    
 10  release_date           41164 non-null  str    
 11  revenue                41184 non-null  float64
 12  runtime                41184 non-null  float64
 13  spoken_languages       41184 non-null  str    
 14  title                  41184 non-null  str    
 15  video             

In [46]:
display(f"Número de linhas após limpeza: {len(df_movies_limpo)}")
display(f"Número de linhas removidas: {len(df_movies) - len(df_movies_limpo)}")

'Número de linhas após limpeza: 41184'

'Número de linhas removidas: 4282'

### Correção de Tipos: Habilitando Análises Financeiras e Temporais

As colunas financeiras e de data chegam do CSV como strings — um efeito colateral da heterogeneidade do dataset original. Sem a conversão para seus tipos nativos, qualquer operação de ordenação temporal, cálculo de ROI ou agregação financeira produziria resultados incorretos ou erros silenciosos.

In [47]:
# Transformando a coluna 'release_date' para datetime
df_movies_limpo['release_date'] = pd.to_datetime(df_movies_limpo['release_date'], errors='coerce')

# Forçando a conversão para numérico (float)
df_movies_limpo['budget'] = pd.to_numeric(df_movies_limpo['budget'], errors='coerce')
df_movies_limpo['popularity'] = pd.to_numeric(df_movies_limpo['popularity'], errors='coerce')

# Tranfromando as colunas 'adult' e 'video' para booleano
df_movies_limpo['adult'] = df_movies_limpo['adult'].astype(str).str.strip().str.lower() == 'true'
df_movies_limpo['video'] = df_movies_limpo['video'].astype(str).str.strip().str.lower() == 'true'

# Verificando o resultado final
df_movies_limpo.info()

<class 'pandas.DataFrame'>
Index: 41184 entries, 0 to 45463
Data columns (total 18 columns):
 #   Column                 Non-Null Count  Dtype         
---  ------                 --------------  -----         
 0   adult                  41184 non-null  bool          
 1   belongs_to_collection  4344 non-null   str           
 2   budget                 41184 non-null  int64         
 3   genres                 41184 non-null  str           
 4   id                     41184 non-null  str           
 5   original_language      41184 non-null  str           
 6   original_title         41184 non-null  str           
 7   popularity             41184 non-null  float64       
 8   production_companies   41184 non-null  str           
 9   production_countries   41184 non-null  str           
 10  release_date           41164 non-null  datetime64[us]
 11  revenue                41184 non-null  float64       
 12  runtime                41184 non-null  float64       
 13  spoken_languages 

### Tratamento de Zeros como Dados Ausentes: Proteção da Integridade Financeira

No contexto deste dataset, valores iguais a zero nas colunas `budget` e `revenue` não representam produções sem custo ou sem receita — representam **ausência de dado declarado**. Mantê-los como zero distorceria qualquer análise de distribuição financeira, cálculo de ROI médio ou correlação entre investimento e desempenho de bilheteria.

- **Regra de negócio:** Zero financeiro neste dataset equivale semanticamente a `NaN` — campo não informado
- **Impacto:** Preserva a validade estatística de todas as métricas financeiras nas análises subsequentes

In [48]:
# Mudando 0 para NaN para não distorcer as análises de orçamento e receita
df_movies_limpo['budget'] = df_movies_limpo['budget'].replace(0, np.nan) 
df_movies_limpo['revenue'] = df_movies_limpo['revenue'].replace(0, np.nan) 

In [49]:
# Conferindo as primeiras linhas do DataFrame limpo
df_movies_limpo.head()

,adult,belongs_to_collection,budget,genres,id,original_language,original_title,popularity,production_companies,production_countries,release_date,revenue,runtime,spoken_languages,title,video,vote_average,vote_count
0,False,"{'id': 10194, 'name': 'Toy Story Collection', ...",30000000.0,"[{'id': 16, 'name': 'Animation'}, {'id': 35, '...",862,en,Toy Story,21.946943,"[{'name': 'Pixar Animation Studios', 'id': 3}]","[{'iso_3166_1': 'US', 'name': 'United States o...",1995-10-30,373554033.0,81.0,"[{'iso_639_1': 'en', 'name': 'English'}]",Toy Story,False,7.7,5415.0
1,False,NaN,65000000.0,"[{'id': 12, 'name': 'Adventure'}, {'id': 14, '...",8844,en,Jumanji,17.015539,"[{'name': 'TriStar Pictures', 'id': 559}, {'na...","[{'iso_3166_1': 'US', 'name': 'United States o...",1995-12-15,262797249.0,104.0,"[{'iso_639_1': 'en', 'name': 'English'}, {'iso...",Jumanji,False,6.9,2413.0
2,False,"{'id': 119050, 'name': 'Grumpy Old Men Collect...",NaN,"[{'id': 10749, 'name': 'Romance'}, {'id': 35, ...",15602,en,Grumpier Old Men,11.712900,"[{'name': 'Warner Bros.', 'id': 6194}, {'name'...","[{'iso_3166_1': 'US', 'name': 'United States o...",1995-12-22,NaN,101.0,"[{'iso_639_1': 'en', 'name': 'English'}]",Grumpier Old Men,False,6.5,92.0
3,False,NaN,16000000.0,"[{'id': 35, 'name': 'Comedy'}, {'id': 18, 'nam...",31357,en,Waiting to Exhale,3.859495,[{'name': 'Twentieth Century Fox Film Corporat...,"[{'iso_3166_1': 'US', 'name': 'United States o...",1995-12-22,81452156.0,127.0,"[{'iso_639_1': 'en', 'name': 'English'}]",Waiting to Exhale,False,6.1,34.0
4,False,"{'id': 96871, 'name': 'Father of the Bride Col...",NaN,"[{'id': 35, 'name': 'Comedy'}]",11862,en,Father of the Bride Part II,8.387519,"[{'name': 'Sandollar Productions', 'id': 5842}...","[{'iso_3166_1': 'US', 'name': 'United States o...",1995-02-10,76578911.0,106.0,"[{'iso_639_1': 'en', 'name': 'English'}]",Father of the Bride Part II,False,5.7,173.0


---
## Salvando Dataset Limpo

In [50]:
# Salvando o dataset limpo para a próxima etapa de análise
caminho_limpo = '../data/processed/movies_cleaned.csv'
df_movies_limpo.to_csv(caminho_limpo, index=False)
print(f"Dataset limpo salvo em: {caminho_limpo}")

Dataset limpo salvo em: ../data/processed/movies_cleaned.csv


---
## Conclusão da Limpeza e Estruturação de Dados

O dataset de filmes passou pelas etapas essenciais de higienização e formatação para garantir a integridade das análises financeiras e de recepção do público. As principais intervenções nesta etapa incluíram:
* **Descarte de colunas de alta cardinalidade de texto**
    - Colunas descartadas: `homepage`, `poster_path`, `overview`, `tagline`, `status` e `imdb_id`.
* **Tratamento de Inconsistências:** Identificação e manejo de valores nulos e registros discrepantes, com atenção especial às métricas críticas de negócio, como `budget` (orçamento) e `revenue` (receita). 
* **Padronização e Tipagem (Data Casting):** Conversão de formatos genéricos para tipos otimizados. Isso incluiu a transformação de indicadores binários para booleanos e a estruturação de datas de lançamento para `datetime`, habilitando futuras análises de séries temporais.

**Exportação:**
O dataset resultante, agora estruturalmente validado, foi exportado com sucesso para `data/processed/movies_cleaned.csv`.

**Próximo Passo:**
Com a base de filmes íntegra e padronizada, o pipeline avança para a **Engenharia de Features** (para criação de variáveis como lucro e extração de dicionários JSON) e, na sequência, para a **Análise Exploratória de Dados (EDA)**, onde investigaremos as dinâmicas de bilheteria e engajamento.

* **Vá para o notebook:** `08_feature_engineering_movies.ipynb`.
---